In [ ]:
# Cell 1: Secrets
import os


In [2]:
# Cell 2: Setup
!git clone https://github.com/sk3feel/hidden-data-reproduction-multimodal.git /content/course_work2026 2>/dev/null || (cd /content/course_work2026 && git pull)
%cd /content/course_work2026
!pip install -q -r requirements.txt
!pip install -q accelerate Pillow pandas tqdm einops timm comet_ml huggingface_hub
!pip install -q peft bitsandbytes trl qwen-vl-utils
!pip uninstall -y transformers tokenizers
!pip install -q tokenizers==0.21.0
!pip install -q transformers==4.49.0 --force-reinstall --no-deps
!python src/download_from_hf.py --repo-id sk3feel/docvqa-privacy-data


/content/course_work2026
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.6/14.6 MB 142.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.3/42.3 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 786.2/786.2 kB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 157.8 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.8/59.8 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.7/76.7 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 91.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jupyter-kernel-gateway 2.5.2 requires notebook<7.0,>=5.7.6, but you have notebook 7.5.5 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [3]:
# Cell 3: Imports and config
import gc, json, os, re, torch, pandas as pd
from pathlib import Path
from collections import defaultdict
from PIL import Image, ImageFile
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from huggingface_hub import HfApi, login
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from qwen_vl_utils import process_vision_info
import comet_ml
import matplotlib.pyplot as plt

import transformers
print(f"transformers version: {transformers.__version__}")
if not transformers.__version__.startswith("4.4"):
    print(f"WARNING: expected 4.4x.x, got {transformers.__version__}. Florence-2 may fail.")

ImageFile.LOAD_TRUNCATED_IMAGES = True

MODEL_CONFIGS = [
    {"name": "Qwen/Qwen2-VL-2B-Instruct", "tag": "qwen2b", "epochs": 20, "batch_size": 1, "checkpoint_epochs": [5, 10, 20], "grad_accum": 4},
    {"name": "Qwen/Qwen2-VL-7B-Instruct", "tag": "qwen7b", "epochs": 10, "batch_size": 1, "checkpoint_epochs": [5, 10], "grad_accum": 4},
]
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
LR = 2e-4
HF_MODEL_REPO = "sk3feel/docvqa-privacy-checkpoints"
TRAIN_SIZE = 200
MAX_IMAGE_SIDE = 1024

login(token=os.environ["HF_TOKEN"])
api = HfApi()
api.create_repo(repo_id=HF_MODEL_REPO, repo_type="model", private=True, exist_ok=True)


transformers version: 4.49.0


Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


RepoUrl('https://huggingface.co/sk3feel/docvqa-privacy-checkpoints', endpoint='https://huggingface.co', repo_type='model', repo_id='sk3feel/docvqa-privacy-checkpoints')

In [4]:
# Cell 4: Load data and stratified subset
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows


def resolve_image_path(raw_path, split):
    path = Path(raw_path)
    if path.exists():
        return path
    image_name = raw_path.replace("\\", "/").split("/")[-1]
    split_dir = "benchmark_train" if split == "train" else "benchmark"
    candidate = Path("artifacts/docqa_recovery") / split_dir / "images/original" / image_name
    if candidate.exists():
        return candidate
    raise FileNotFoundError(f"Image not found: {raw_path}")


def extract_question(row):
    if row.get("question"):
        return str(row["question"]).strip()
    chat = row.get("chat_messages") or []
    if chat:
        for item in chat[0].get("content", []):
            if item.get("type") == "text":
                match = re.search(r"Question:\s*(.*)", item["text"], re.IGNORECASE | re.DOTALL)
                return match.group(1).strip() if match else item["text"].strip()
    return str(row.get("prompt", "")).strip()


def prepare_records(path, default_split):
    rows = load_jsonl(path)
    records = []
    for row in rows:
        split = str(row.get("split", default_split))
        records.append({
            "example_id": str(row.get("example_id", "")),
            "image_path": str(resolve_image_path(str(row["image_path"]), split)),
            "question": extract_question(row),
            "answer": str(row["answer"]).strip(),
            "split": split,
        })
    return records


def enrich_with_field_types(records, manifest_path):
    manifest = load_jsonl(manifest_path)
    type_map = {}
    for row in manifest:
        eid = str(row.get("example_id", row.get("question_id", "")))
        type_map[eid] = row.get("coarse_field_type", "OTHER")
    for rec in records:
        rec["coarse_field_type"] = type_map.get(rec["example_id"], "OTHER")
    return records


def stratified_sample(records, n, type_key="coarse_field_type"):
    by_type = defaultdict(list)
    for rec in records:
        by_type[rec[type_key]].append(rec)
    types = sorted(by_type.keys())
    per_type = n // len(types)
    result = []
    for ft in types:
        result.extend(by_type[ft][:per_type])
    used = {r["example_id"] for r in result}
    for rec in records:
        if len(result) >= n:
            break
        if rec["example_id"] not in used:
            result.append(rec)
    return result[:n]


all_train = prepare_records("artifacts/finetuning_generative/train_qwen2vl.jsonl", "train")
all_train = enrich_with_field_types(all_train, "artifacts/docqa_recovery/benchmark_train/manifest.jsonl")
validation_records = prepare_records("artifacts/finetuning_generative/validation_qwen2vl.jsonl", "validation")
validation_records = enrich_with_field_types(validation_records, "artifacts/docqa_recovery/benchmark/manifest.jsonl")

train_records = stratified_sample(all_train, TRAIN_SIZE)

type_dist = defaultdict(int)
for rec in train_records:
    type_dist[rec["coarse_field_type"]] += 1
print(f"Train: {len(train_records)} records")
print(f"Distribution: {dict(type_dist)}")
print(f"Validation: {len(validation_records)} records")

train_ids = [r["example_id"] for r in train_records]
pd.DataFrame({"example_id": train_ids}).to_csv("artifacts/finetuning_generative/qwen_train_200_ids.csv", index=False)


Train: 200 records
Distribution: {'AMOUNT': 35, 'CONTACT_ADR': 33, 'DATE': 33, 'ID': 33, 'ORG': 33, 'PERSON': 33}
Validation: 1612 records


In [5]:
# Cell 5: Functions
def safe_open_image(path, max_side=MAX_IMAGE_SIDE):
    image = Image.open(path).convert("RGB")
    if max(image.size) > max_side:
        image.thumbnail((max_side, max_side), Image.LANCZOS)
    return image


def make_user_text(question):
    return f"Answer the question about this document image. Give only the answer value, nothing else.\nQuestion: {question}"


def build_messages(record, image):
    return [
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": make_user_text(record["question"])},
        ]},
        {"role": "assistant", "content": [
            {"type": "text", "text": record["answer"]},
        ]},
    ]


def load_qwen_for_training(model_name):
    bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_quant_type="nf4")
    model = Qwen2VLForConditionalGeneration.from_pretrained(model_name, quantization_config=bnb, device_map="auto", trust_remote_code=True)
    processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
    model = prepare_model_for_kbit_training(model)
    model.gradient_checkpointing_enable()
    if hasattr(model, "enable_input_require_grads"):
        model.enable_input_require_grads()
    lora = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, target_modules=["q_proj", "v_proj"], lora_dropout=LORA_DROPOUT, bias="none", task_type="CAUSAL_LM")
    model = get_peft_model(model, lora)
    model.print_trainable_parameters()
    return model, processor


def cleanup_gpu():
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU free: {torch.cuda.mem_get_info()[0]/1024**3:.1f} GB")


def make_collate_fn(processor):
    def collate_fn(batch):
        prompt_texts, full_texts, prompt_images, full_images = [], [], [], []
        for rec in batch:
            image = safe_open_image(rec["image_path"])
            prompt_msgs = build_messages(rec, image)[:1]
            full_msgs = build_messages(rec, image)
            prompt_texts.append(processor.apply_chat_template(prompt_msgs, tokenize=False, add_generation_prompt=True))
            full_texts.append(processor.apply_chat_template(full_msgs, tokenize=False, add_generation_prompt=False))
            p_img, _ = process_vision_info(prompt_msgs)
            f_img, _ = process_vision_info(full_msgs)
            prompt_images.append(p_img[0] if isinstance(p_img, list) else p_img)
            full_images.append(f_img[0] if isinstance(f_img, list) else f_img)
        prompt_batch = processor(text=prompt_texts, images=prompt_images, return_tensors="pt", padding=True)
        full_batch = processor(text=full_texts, images=full_images, return_tensors="pt", padding=True)
        labels = full_batch["input_ids"].clone()
        labels[full_batch["attention_mask"] == 0] = -100
        for idx, pl in enumerate(prompt_batch["attention_mask"].sum(dim=1).tolist()):
            labels[idx, :int(pl)] = -100
        result = {k: v for k, v in full_batch.items()}
        result["labels"] = labels
        return result
    return collate_fn


def normalize_text(text):
    return " ".join(str(text).strip().lower().split())


def run_sanity_check(model, processor, records, n=20):
    model.eval()
    correct = 0
    device = next(model.parameters()).device
    for rec in records[:n]:
        image = safe_open_image(rec["image_path"])
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": make_user_text(rec["question"])},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        img_inputs, _ = process_vision_info(messages)
        img_input = img_inputs[0] if isinstance(img_inputs, list) else img_inputs
        inputs = processor(text=[text], images=[img_input], return_tensors="pt", padding=True)
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=50)
        pred = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
        if normalize_text(pred) == normalize_text(rec["answer"]):
            correct += 1
    model.train()
    return correct / max(n, 1)


def collect_sanity_rows(model, processor, records, split_name, n=100):
    model.eval()
    rows = []
    correct = 0
    device = next(model.parameters()).device
    for rec in records[:n]:
        image = safe_open_image(rec["image_path"])
        messages = [{"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": make_user_text(rec["question"])},
        ]}]
        text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        img_inputs, _ = process_vision_info(messages)
        img_input = img_inputs[0] if isinstance(img_inputs, list) else img_inputs
        inputs = processor(text=[text], images=[img_input], return_tensors="pt", padding=True)
        inputs = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in inputs.items()}
        with torch.no_grad():
            gen = model.generate(**inputs, max_new_tokens=50)
        pred = processor.batch_decode(gen[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)[0].strip()
        is_match = normalize_text(pred) == normalize_text(rec["answer"])
        correct += int(is_match)
        rows.append({
            "split": split_name,
            "example_id": rec["example_id"],
            "question": rec["question"],
            "gold": rec["answer"],
            "prediction": pred,
            "exact_match": is_match,
            "coarse_field_type": rec.get("coarse_field_type", "OTHER"),
        })
    model.train()
    return rows, correct / max(len(rows), 1)


In [6]:
# Cell 6: Training loop
gc.collect()
torch.cuda.empty_cache()


class SimpleDataset(Dataset):
    def __init__(self, records): self.records = records
    def __len__(self): return len(self.records)
    def __getitem__(self, idx): return self.records[idx]


for config in MODEL_CONFIGS:
    tag = config["tag"]
    print(f"\n{'='*60}\nTraining {tag}: {config['name']}\n{'='*60}")

    experiment = comet_ml.Experiment(api_key=os.environ["COMET_API_KEY"], workspace="scfeel", project_name="qwen3-1")
    experiment.set_name(f"{tag}-finetune")
    experiment.log_parameters({**config, "lora_r": LORA_R, "lr": LR, "train_size": TRAIN_SIZE})

    model, processor = load_qwen_for_training(config["name"])
    dataset = SimpleDataset(train_records)
    dataloader = DataLoader(dataset, batch_size=config["batch_size"], shuffle=True, collate_fn=make_collate_fn(processor))
    optimizer = torch.optim.AdamW((p for p in model.parameters() if p.requires_grad), lr=LR)
    train_loss_history = []

    global_step = 0
    for epoch in range(1, config["epochs"] + 1):
        model.train()
        optimizer.zero_grad()
        epoch_loss, valid_batches = 0.0, 0
        for batch in tqdm(dataloader, desc=f"{tag} Epoch {epoch}/{config['epochs']}"):
            device = next(model.parameters()).device
            batch = {k: v.to(device) if isinstance(v, torch.Tensor) else v for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                outputs = model(**batch)
                loss = outputs.loss / config["grad_accum"]
            if not loss.isfinite():
                optimizer.zero_grad()
                continue
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            if (valid_batches + 1) % config["grad_accum"] == 0:
                optimizer.step()
                optimizer.zero_grad()
                global_step += 1
            epoch_loss += float(loss.item()) * config["grad_accum"]
            valid_batches += 1
        if valid_batches % config["grad_accum"] != 0:
            optimizer.step()
            optimizer.zero_grad()
        avg_loss = epoch_loss / max(valid_batches, 1)
        train_loss_history.append({"epoch": epoch, "train_loss": avg_loss})
        print(f"{tag} epoch {epoch}, loss: {avg_loss:.4f}")
        experiment.log_metric("train_loss", avg_loss, epoch=epoch)

        if epoch in config["checkpoint_epochs"]:
            ckpt = f"checkpoints/{tag}/epoch_{epoch}"
            os.makedirs(ckpt, exist_ok=True)
            model.save_pretrained(ckpt)
            processor.save_pretrained(ckpt)
            api.upload_folder(folder_path=ckpt, repo_id=HF_MODEL_REPO, path_in_repo=f"{tag}/epoch_{epoch}", repo_type="model")
            em = run_sanity_check(model, processor, train_records, n=20)
            print(f"  Sanity EM@{epoch}: {em:.2f}")
            experiment.log_metric("sanity_em", em, epoch=epoch)

    seen_rows, seen_em = collect_sanity_rows(model, processor, train_records, "seen", n=min(100, len(train_records)))
    unseen_rows, unseen_em = collect_sanity_rows(model, processor, validation_records, "unseen", n=100)
    print(f"{tag}: seen_em={seen_em:.2f}, unseen_em={unseen_em:.2f}")
    experiment.log_metric("final_seen_em", seen_em)
    experiment.log_metric("final_unseen_em", unseen_em)

    sanity_final_df = pd.DataFrame(seen_rows + unseen_rows)
    sanity_path = Path("artifacts/finetuning_generative") / f"{tag}_sanity_final.csv"
    sanity_final_df.to_csv(sanity_path, index=False)
    experiment.log_table(f"{tag}_sanity_final.csv", sanity_final_df)
    experiment.log_asset(str(sanity_path), file_name=sanity_path.name)

    train_loss_df = pd.DataFrame(train_loss_history)
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.plot(train_loss_df["epoch"], train_loss_df["train_loss"], marker="o")
    ax.set_title(f"{tag} train loss ?? ??????")
    ax.set_xlabel("?????")
    ax.set_ylabel("Train loss")
    ax.grid(alpha=0.3)
    plt.tight_layout()
    experiment.log_figure(f"{tag}_train_loss_curve", fig)
    plt.close(fig)

    experiment.end()

    del model, processor, dataset, dataloader, optimizer
    cleanup_gpu()


COMET WARNING: To get all data logged automatically, import comet_ml before the following modules: torch, tensorflow, sklearn, keras.
COMET WARNING: As you are running in a Jupyter environment, you will need to call `experiment.end()` when finished to ensure all metrics and code are logged before exiting.



Training qwen2b: Qwen/Qwen2-VL-2B-Instruct


COMET INFO: Experiment is live on comet.com https://www.comet.com/scfeel/qwen3-1/21563b3c5e3f4230890ece5b0aff925a

The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

...

qwen7b epoch 1, loss: 1.5416


qwen7b Epoch 2/5:   0%|          | 0/200 [00:00<?, ?it/s]

qwen7b epoch 2, loss: 1.3540


qwen7b Epoch 3/5:   0%|          | 0/200 [00:00<?, ?it/s]

KeyboardInterrupt: 